In [ ]:
# ============================================================
# ADVANCED PREPROCESSING PIPELINE
# DOES NOT REMOVE SAMPLES
# GOOGLE COLAB READY
# ============================================================

# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

!apt-get install unrar -y
!pip install pandas scikit-learn openpyxl xlrd beautifulsoup4 -q

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import re
import html
import pandas as pd

from bs4 import BeautifulSoup

from google.colab import files
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# STEP 1: UPLOAD FILES
# ============================================================

print("Upload ALL files together:")
print("1. ironic.rar")
print("2. regular.rar")
print("3. pairings.txt OR pairings.csv")
print("4. sarcasm_lines.txt")
print("5. file_labels.xls/xlsx")

uploaded = files.upload()

# ============================================================
# STEP 2: CREATE FOLDERS
# ============================================================

!mkdir -p ironic
!mkdir -p regular

# ============================================================
# STEP 3: EXTRACT RAR FILES
# ============================================================

for file in uploaded.keys():

    lower = file.lower()

    if "ironic" in lower and file.endswith(".rar"):

        !unrar x -o+ "{file}" ironic/

    elif "regular" in lower and file.endswith(".rar"):

        !unrar x -o+ "{file}" regular/

print("RAR extraction completed!")

# ============================================================
# STEP 4: ADVANCED CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # --------------------------------------------------------
    # SAFETY
    # --------------------------------------------------------

    if pd.isna(text):
        return ""

    text = str(text)

    # --------------------------------------------------------
    # HTML ENTITY DECODING
    # --------------------------------------------------------

    text = html.unescape(text)

    # --------------------------------------------------------
    # REMOVE HTML TAGS
    # --------------------------------------------------------

    text = BeautifulSoup(text, "html.parser").get_text()

    # --------------------------------------------------------
    # LOWERCASE
    # --------------------------------------------------------

    text = text.lower()

    # --------------------------------------------------------
    # REMOVE URLS
    # --------------------------------------------------------

    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )

    # --------------------------------------------------------
    # REMOVE EMAILS
    # --------------------------------------------------------

    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    # --------------------------------------------------------
    # REMOVE USERNAMES
    # --------------------------------------------------------

    text = re.sub(
        r'@\w+',
        ' ',
        text
    )

    # --------------------------------------------------------
    # NORMALIZE CONTRACTIONS
    # --------------------------------------------------------

    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'s": " is",
        "'d": " would",
        "'ll": " will",
        "'t": " not",
        "'ve": " have",
        "'m": " am"
    }

    for key, value in contractions.items():

        text = text.replace(key, value)

    # --------------------------------------------------------
    # REMOVE SPECIAL CHARACTERS
    # KEEP SARCASM PUNCTUATION
    # --------------------------------------------------------

    text = re.sub(
        r"[^a-zA-Z0-9!?.,' ]",
        " ",
        text
    )

    # --------------------------------------------------------
    # REMOVE REPEATED PUNCTUATION
    # --------------------------------------------------------

    text = re.sub(
        r'([!?.,])\1+',
        r'\1',
        text
    )

    # --------------------------------------------------------
    # NORMALIZE REPEATED LETTERS
    # ex: soooo -> soo
    # --------------------------------------------------------

    text = re.sub(
        r'(.)\1{3,}',
        r'\1\1',
        text
    )

    # --------------------------------------------------------
    # REMOVE EXTRA SPACES
    # --------------------------------------------------------

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

# ============================================================
# STEP 5: READ ALL TXT FILES
# ============================================================

def read_reviews(folder):

    reviews = {}

    for root, dirs, files_list in os.walk(folder):

        for file in files_list:

            if file.endswith(".txt"):

                path = os.path.join(root, file)

                try:

                    with open(
                        path,
                        "r",
                        encoding="utf-8",
                        errors="ignore"
                    ) as f:

                        text = f.read()

                        text = clean_text(text)

                        reviews[file] = text

                except Exception as e:

                    print("Error:", path)

    return reviews

# ============================================================
# STEP 6: LOAD REVIEWS
# ============================================================

print("Loading ironic reviews...")
ironic_reviews = read_reviews("ironic")

print("Loading regular reviews...")
regular_reviews = read_reviews("regular")

print("Ironic files:", len(ironic_reviews))
print("Regular files:", len(regular_reviews))

# ============================================================
# STEP 7: LOAD LABEL FILE
# ============================================================

labels_file = None

for file in uploaded.keys():

    if "label" in file.lower():

        labels_file = file
        break

labels_df = None

if labels_file:

    try:

        labels_df = pd.read_excel(labels_file)

        print("\nLabels file loaded!")

    except Exception as e:

        print("Error loading labels file:", e)

# ============================================================
# STEP 8: FIND PAIRING FILE
# ============================================================

pair_file = None

for file in uploaded.keys():

    if "pair" in file.lower():

        pair_file = file
        break

print("\nPairing file:", pair_file)

# ============================================================
# STEP 9: CREATE DATASET
# ============================================================

dataset = []

if pair_file:

    pairs = pd.read_csv(
        pair_file,
        sep=None,
        engine="python",
        header=None
    )

    print("\nPairings loaded!")

    for i, row in pairs.iterrows():

        try:

            ironic_name = str(row[0]).strip()
            regular_name = str(row[1]).strip()

            # ------------------------------------------------
            # IRONIC REVIEW
            # ------------------------------------------------

            if ironic_name in ironic_reviews:

                dataset.append({
                    "text": ironic_reviews[ironic_name],
                    "label": 1,
                    "group": i,
                    "file_name": ironic_name
                })

            # ------------------------------------------------
            # REGULAR REVIEW
            # ------------------------------------------------

            if regular_name in regular_reviews:

                dataset.append({
                    "text": regular_reviews[regular_name],
                    "label": 0,
                    "group": i,
                    "file_name": regular_name
                })

        except:
            pass

# ============================================================
# STEP 10: FALLBACK
# ============================================================

if len(dataset) < 100:

    print("\nUsing all reviews individually...")

    for k, v in ironic_reviews.items():

        dataset.append({
            "text": v,
            "label": 1,
            "group": k,
            "file_name": k
        })

    for k, v in regular_reviews.items():

        dataset.append({
            "text": v,
            "label": 0,
            "group": k,
            "file_name": k
        })

# ============================================================
# STEP 11: ADD sarcasm_lines.txt
# ============================================================

for file in uploaded.keys():

    if "sarcasm" in file.lower() and file.endswith(".txt"):

        with open(
            file,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            for i, line in enumerate(f):

                line = clean_text(line)

                if len(line) > 0:

                    dataset.append({
                        "text": line,
                        "label": 1,
                        "group": f"sarcasm_{i}",
                        "file_name": f"sarcasm_{i}"
                    })

# ============================================================
# STEP 12: CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(dataset)

# ============================================================
# STEP 13: FINAL TEXT NORMALIZATION
# ============================================================

df["text"] = df["text"].astype(str)

df["text"] = df["text"].apply(clean_text)

# ============================================================
# STEP 14: HANDLE EMPTY TEXT
# DO NOT REMOVE SAMPLES
# ============================================================

df["text"] = df["text"].replace("", "[empty_review]")

# ============================================================
# STEP 15: REMOVE EXACT DUPLICATES
# KEEP FIRST OCCURRENCE
# ============================================================

df = df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

# ============================================================
# STEP 16: TEXT LENGTH FEATURES
# ============================================================

df["char_length"] = df["text"].apply(len)

df["word_length"] = df["text"].apply(
    lambda x: len(x.split())
)

# ============================================================
# STEP 17: SHUFFLE DATASET
# ============================================================

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ============================================================
# STEP 18: SHOW DATASET INFO
# ============================================================

print("\n========================")
print("FINAL DATASET")
print("========================")

print("Total Samples:", len(df))

print("\nLabel Distribution:")
print(df["label"].value_counts())

print("\nAverage Words:")
print(df["word_length"].mean())

print("\nAverage Characters:")
print(df["char_length"].mean())

# ============================================================
# STEP 19: GROUP-AWARE SPLITTING
# ============================================================

X = df["text"]
y = df["label"]
groups = df["group"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(X, y, groups)
)

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(
        temp_df["text"],
        temp_df["label"],
        temp_df["group"]
    )
)

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

# ============================================================
# STEP 20: SAVE CSV FILES
# ============================================================

train_df.to_csv(
    "train.csv",
    index=False
)

val_df.to_csv(
    "validation.csv",
    index=False
)

test_df.to_csv(
    "test.csv",
    index=False
)

df.to_csv(
    "full_dataset.csv",
    index=False
)

print("\nCSV files saved successfully!")

# ============================================================
# STEP 21: DOWNLOAD FILES
# ============================================================

files.download("train.csv")
files.download("validation.csv")
files.download("test.csv")
files.download("full_dataset.csv")

# ============================================================
# STEP 22: PREVIEW
# ============================================================

df.head()

In [ ]:
train_df.to_csv("train.csv", index=False)

val_df.to_csv("validation.csv", index=False)

test_df.to_csv("test.csv", index=False)

df.to_csv("full_dataset.csv", index=False)

print("\nCSV files saved successfully!")

# ============================================================
# STEP 21: DOWNLOAD FILES
# ============================================================

files.download("train.csv")
files.download("validation.csv")
files.download("test.csv")
files.download("full_dataset.csv")